# Data pipeline journal: from sources to one merged table

*Personal notes by Deric, written to review later. Not part of the graded analysis
(`main.ipynb`); the data description there is Adam's part.*

This notebook retraces, step by step, how the data side of the project was built:

1. **Retrieval**: getting each source into a clean file (`src/data/*.py`)
2. **Validation**: proving the files are right, and *on time* (why this counts as validation)
3. **Cleaning decisions**: the rules every source follows
4. **Merging**: one daily table (`src/cleaning.py`)
5. **Git**: branches, an accident on `main`, merge conflicts, rewriting history
6. **Protocols**: the checklists to reuse next time

Each part ends with a **Lesson** box: the situation, how it was resolved, the protocol,
the intuition and the design behind it.

| Date | Step | Where |
|------|------|-------|
| 13–19 Oct 2026 | first retrieval modules, CFTC release-date rule | `feat/*`, `cftc-check-pre-clean` |
| 1–2 Nov | BIS, Yahoo, MoF, FRED validated on download + tests | `fix/bis`, `fix/yahoo`, `fix/mof`, `fix/fred` |
| 3 Nov | open questions settled (index rule, lags, JP break, shutdowns) | `final-validation` → PR #23 |
| 3–4 Nov | FRED commit pushed to `main` by mistake, reverted, reapplied | PR #24 (`reapply-fred`) |
| 4–5 Nov | merged daily table | `data-cleaning` → PR #25 |

In [ ]:
import pandas as pd

from src import config
from src.cleaning import OUTPUT_FILE, load_sources

sources = load_sources()      # the five saved files from src/data/*
daily = pd.read_parquet(OUTPUT_FILE)
{name: df.shape for name, df in sources.items()}

## 1. Retrieval: one module per source

Every source has the same shape of module: a `fetch_*` / `download_*` function that
**returns** a DataFrame, a `save_*` function, and an `if __name__ == "__main__":` block.
Settings (URLs, tickers, series ids, lags, window) all live in `src/config.py`, so a
change is made in one place.

| Source | Module | What it gives | Traps met on the way |
|--------|--------|---------------|----------------------|
| Yahoo Finance | `yahoo.py` | daily closes of 9 assets | one-day bad prints in USD/JPY (e.g. 109.24 between 92.82 and 92.30 in 2008); `end` is **excluded** by yfinance |
| FRED (OECD) | `fred.py` | US and JP 3-month rates, monthly, % | columns renamed by *position* could swap US and JP; JP series changes source in 2025-01 |
| CFTC | `cftc.py` | weekly speculative yen positioning | one zip for 1986–2016 then one per year; market *name* changed over time, so filter on the contract **code** `097741`; leading zero lost if read as a number |
| BIS | `bis.py` | yen credit to non-residents, quarterly | units: `UNIT_MEASURE = JPY`, `UNIT_MULT = 6` (millions) → divide by 1e6 for trillions; past quarters get **revised** |
| MoF Japan | `mof.py` | FX intervention days | Shift-JIS encoding (`cp932`), bilingual layout, year/month written only on the first row of a block, non-yen pairs in 1991 and 1997–98 |

### Design choices common to all modules
- **Return, then save.** `fetch_cftc()` used to save inside the function; it was moved to
  the `__main__` block so tests and notebooks can call the function without side effects.
- **Rename by key, never by position.** `fred.py` maps `{series_id: name}`; a list of
  names in the order FRED happens to return columns can silently swap US and JP.
- **Run as a module or a file.** `python -m src.data.bis` always works;
  `python src/data/bis.py` only works after `pip install -e .` (Adam's `pyproject.toml`).
  The "No module named 'src'" error came back on my machine only because that install
  had never been run in my `.venv`.

> **Lesson 1: retrieval**
>
> - **Situation:** each source has its own format quirks, and they fail *silently*
>   (a swapped column, a dropped leading zero, a missing last day).
> - **Resolution:** identify things by stable keys (series id, contract code), not by
>   position or label; read the documentation of the API's edge cases (yfinance `end`).
> - **Protocol:** for a new source, (1) print the raw file once, (2) write down units,
>   encoding and keys, (3) put every constant in `config.py`, (4) return a DataFrame
>   with a named index, (5) save in a separate function.
> - **Intuition:** the dangerous errors are not the ones that crash, but the ones that
>   produce a plausible-looking table.
> - **Design:** one module per source, same structure everywhere, so a reviewer who
>   knows one module knows them all.

## 2. Validation: proving the data is right *and on time*

"Validation" here means three different things. Seeing them separately is what made
the work systematic.

| Layer | Question | How it is checked | Example |
|-------|----------|-------------------|---------|
| **Integrity** | Is the file what I think it is? | checks **on download** that raise an error | BIS units and duplicates; FRED all-NaN series, month starts, rates in −2%…25%; MoF 9 columns, dates in order |
| **Content** | Are the values plausible and internally consistent? | `tests/test_*.py` on the saved files | MoF days add up to MoF's own quarterly totals (gap ≤ 5 ×100m yen); CFTC July 2024 peak below −180,000 contracts; no month-on-month FRED jump above 2pp |
| **Timing** | *When* could an investor have known each value? | compare our `release_date` rules with real release events | BIS 2026-Q2 out on 29 Oct (rule: 30 Oct); FRED update of 15 Oct had US to Sep, JP to Aug; CFTC shutdown calendars |

**Why timing is validation, not just cleaning.** A backtest that uses a value before
it was published looks better than reality (*look-ahead bias*). The data can be
perfectly correct and still invalidate the results if it is used on the wrong day.
So each `release_date` rule is a **claim** that had to be checked against evidence.

### 2.1 Checks on download: fail loudly
The modules raise `ValueError` instead of fixing things quietly. Example from `bis.py`:

```python
if raw["UNIT_MEASURE"].iloc[0] != "JPY" or raw["UNIT_MULT"].iloc[0] != 6:
    raise ValueError("BIS units changed, the conversion to trillions is wrong")
```

If BIS ever switches to billions, the code stops instead of producing numbers 1000×
too large.

### 2.2 Tests on saved files
`tests/` checks the files in `data/`. A test module **skips** (does not fail) when its
file was never fetched, so a teammate who only works on one source is not blocked.
Run `python -m pytest -q` after every fetch.

In [ ]:
# Content check example: MoF daily amounts vs the July 2024 episode
mof = sources["mof"]
mof.loc["2024-04-01":"2024-07-31"]

### 2.3 Timing checks, one by one

**FRED release lag (3 months after month start).**
On 3 Nov 2026 the FRED page said "Updated 15 Oct 2026", with US data up to September and
JP up to August, i.e. about 1.5 and 2.5 months after the month starts. Three months is on
the safe side. Past release dates (ALFRED vintages) need an API key and were not checked.

**BIS release lag (4 months after quarter end).**
2026-Q2 (quarter end 30 Jun) was released on 29 Oct 2026; our rule gives 30 Oct, one day
*after* the real release, which is the safe direction.

**CFTC shutdown windows.** The first windows ended when the shutdowns ended. Checking the
CFTC calendar showed reports stayed late until CFTC was *back on schedule*:

| Shutdown | Old window (report dates) | Evidence | New window |
|----------|---------------------------|----------|------------|
| Oct 2013 | 1 → 15 Oct | catch-up from 25 Oct, back on schedule 8 Nov | 1 → 29 Oct |
| Dec 2018 – Jan 2019 | 25 Dec → 29 Jan | first late report due 28 Dec (measured **Monday 24 Dec**, Christmas week), two reports a week from 1 Feb, back on schedule 8 Mar | 24 Dec → 26 Feb |

The old windows missed 22 and 29 Oct 2013, 24 Dec 2018 and all of February 2019.

In [ ]:
cftc = sources["cftc"]
cftc.loc["2018-12-15":"2019-03-10", ["release_date", "delayed_release"]]

### 2.4 The JP rate break: a validation story

**Symptom.** From 2025-01, FRED `jp_3m` switched from 3 to 5 decimals and jumped
+38bp in one month, while the BoJ only hiked 25bp, on 24 January. A monthly average
should move *less* than the hike, not more.

**Hypothesis 1 (from a teammate): only rounding changed.** Rejected: rounding to 3
decimals moves a value by at most 0.0005pp, not 0.38pp.

**Hypothesis 2: the underlying rate changed.** Values after 2025 are multiples of 1/22
(0.77273 = 17/22) and stay flat for months, the pattern of a quoted, sticky rate.

**Test.** The JBA publishes daily JBA TIBOR only as yearly PDFs. Parsing the 2024 and
2025 PDFs (3-month column) and comparing with FRED:

| Period | FRED vs **month-end** 3-month JBA TIBOR |
|--------|-------------------------------------------|
| Jan–Dec 2025 | **exactly equal**, 12 months out of 12 |
| Jan–Dec 2024 | FRED 6–23bp **lower** every month |

The first comparison used the monthly *average* of TIBOR and gave near-matches; switching
to the *month-end* value gave exact matches, which also showed that from 2025 the value
is not an average any more.

**Decision.** Keep FRED (one source for both rates) and treat 2025-01 as a break
(`config.JP_3M_BREAK`, test in `tests/test_fred.py`); run a robustness check without the
Dec 2024 → Jan 2025 change. Probable old source: Euroyen TIBOR, discontinued on
30 Dec 2024 (not confirmed).

In [ ]:
fred = sources["fred"]
jp = fred["jp_3m"]
pd.DataFrame({"jp_3m": jp, "change": jp.diff()}).loc["2024-09-01":"2025-04-01"]

> **Lesson 2: validation**
>
> - **Situation:** data that *looks* fine (right shape, no NaN) can still be wrong in
>   units, content or timing; and a release-date rule is a guess until checked.
> - **Resolution:** three layers: checks on download (integrity), tests on saved files
>   (content), and comparison with external evidence (timing): release calendars, an
>   independent source (JBA TIBOR), the source's own totals (MoF quarterly sums).
> - **Protocol:** for every number the analysis relies on, ask "what would prove this
>   wrong?" and write that as a check. When something looks odd, write the hypotheses
>   down, find the cheapest test that separates them, and keep the evidence in
>   `docs/decisions.md`.
> - **Intuition:** an anomaly with a *size* is a clue: 38bp vs a 25bp hike ruled out
>   rounding immediately. Compare exact values before averages; an exact match is much
>   stronger evidence than a close one.
> - **Design:** checks that **raise** errors live in the download code; checks that
>   **describe** expected properties live in `tests/`; estimates are named constants
>   in `config.py` with a comment saying how they were checked.

## 3. Cleaning decisions: one rule for every source

### 3.1 The index rule (and why PR #20 was undone)
PR #20 (Adam) changed the BIS index from the quarter-end date to `release_date`.
It works for BIS alone, but not as a rule for all sources:

1. **MoF cannot be indexed by release date.** All intervention days of a quarter share
   one release date (quarter end + 45 days): 11 and 12 July 2024 would both land on
   14 Nov 2024, giving duplicates or losing the daily detail.
2. **The event study needs the event date.** USD/JPY reacted on 11–12 July, not in November.
3. **Release dates are estimates.** If a lag is corrected later, only one column changes,
   not the index of every table built on it.

**Rule adopted:** index = the date the value refers to (intervention day, quarter end,
month start, CFTC report Tuesday); `release_date` is a column. CFTC moved from a
release-date index to `report_date`; BIS went back to quarter end.

### 3.2 Which date for which analysis
| Analysis | Aligns on | Why |
|----------|-----------|-----|
| Event study | the index (event date) | markets react on the day |
| Predictive tests (H2, H3) | `release_date`, via the `_known` columns | only what was public may be used |
| Carry accrual | rate of the day's own month | the investor earns today's rate, even if it is published later |

### 3.3 Other rules
- **No filling of prices.** A closed market stays NaN; filling would create fake zero returns.
- **Bad prints removed in USD/JPY only** (`remove_spikes`): a >3% jump away from both
  neighbours, while the neighbours stay within 2% of each other. Not applied to VIX,
  Bitcoin, oil or USD/MXN, where big reversing moves are real.
- **Gaps stay NaN** (`jp_3m` before 2002-04, `us_3m` 2020-04).
- **Fixed window.** `END = date.today()` changed results on every run; now
  `END = "2026-09-30"`, and `ANALYSIS_START = "2002-04-01"` for tests that need the rate gap.

> **Lesson 3: cleaning rules**
>
> - **Situation:** a rule that is right for one dataset (BIS by release date) broke
>   another (MoF) and the event study; I had approved it before seeing the merge step.
> - **Resolution:** step back to a rule that works for *every* source, keep the other
>   information as a column, and explain the change to the author openly.
> - **Protocol:** before approving a convention, test it against the hardest source and
>   the hardest use case (here: MoF and the event study). Log every rule in
>   `docs/decisions.md` with its reason and, when replaced, strike it through rather
>   than deleting it.
> - **Intuition:** the index answers *what the value is*; `release_date` answers *when
>   it may be used*. Mixing the two in one field is what caused the conflict.
> - **Design:** store facts (event dates) as the index, store estimates (release dates)
>   as columns, and let each analysis choose.

## 4. Merging: one daily table (`src/cleaning.py`)

**Output:** `data/merged_daily.parquet` (moved from `data/processed/` after review),
one row per USD/JPY trading day.

### 4.1 The calendar
The carry trade is priced on USD/JPY, so the table uses **USD/JPY trading days**.
Weekends exist only for Bitcoin and are dropped, so Bitcoin's Friday→Monday change
includes the weekend. Other markets' holidays stay NaN.

In [ ]:
print(f"{len(daily):,} rows, {daily.index[0].date()} to {daily.index[-1].date()}")
daily.notna().mean().round(3).to_frame("share of days with a value")

### 4.2 `as_known`: what was public on each day
```python
known = df.set_index("release_date")[columns].sort_index()
known = known[~known.index.duplicated(keep="last")]
known = known.ffill()                                  # a NaN release brings nothing new
known.index = known.index + pd.Timedelta(days=1)       # usable from the next day only
return known.reindex(calendar, method="ffill")         # latest release before each day
```
Three ideas, each fixing a concrete risk:
1. **Index by `release_date`**, so values appear when they became public.
2. **`ffill()` before reindexing.** FRED `us_3m` is NaN for 2020-04; without this line,
   `us_3m_known` was NaN for ~22 days in July 2020, although the March value was still
   the latest public one. Found by making a test check every 5th day instead of every 250th.
3. **+1 day.** Release dates are estimates and data comes out *during* the day (CFTC at
   15:30 New York time, after Tokyo and London close). Using a value only from the next
   trading day removes same-day look-ahead.

A toy example:

In [ ]:
from src.cleaning import as_known

toy = pd.DataFrame({
    "release_date": pd.to_datetime(["2024-07-05", "2024-07-12", "2024-07-19"]),  # Fridays
    "x": [1.0, None, 3.0],                                                       # 2nd release missing
})
calendar = pd.bdate_range("2024-07-04", "2024-07-23")
as_known(toy, ["x"], calendar).T
# Friday 5 Jul -> still NaN (released that day); Monday 8 Jul -> 1.0
# 15 Jul -> still 1.0 (missing release); Monday 22 Jul -> 3.0

### 4.3 `rates_as_of`: the rate of the day's own month
```python
by_month = rates[RATE_COLUMNS].copy()
by_month.index = by_month.index.to_period("M")                  # 2024-07-01 -> 2024-07
return by_month.reindex(calendar.to_period("M")).set_axis(calendar)
```
Matching on the **month** (not `ffill`) means a month FRED has not published yet stays
NaN instead of repeating the last value: `jp_3m` ends a month before `us_3m`, and
`ffill` would have invented September 2026.

### 4.4 CFTC and MoF
- **CFTC:** reports delayed by a shutdown are dropped from the `_known` columns; the last
  on-time report is carried forward instead (their real release date is unknown). A Friday
  report is used from Monday, so a Friday decision uses the previous week's report.
- **MoF:** amount on its day, 0 otherwise. The build **stops** if an intervention falls on
  a day without a USD/JPY close, so no intervention is lost silently; a test checks that
  the total in the table equals the MoF file. (A `MOF_COVERAGE_END` cut-off was tried and
  dropped: days after the last published quarter show 0.)

In [ ]:
daily.loc["2024-07-08":"2024-07-16", ["usdjpy", "intervention_bn_jpy", "spec_net_known", "rate_diff", "rate_diff_known"]]

### 4.5 Tests of the merge (`tests/test_cleaning.py`)
One row per trading day; inside the window; prices not filled; `_known` value on each day
equals the latest release *strictly before* that day; CFTC usable on the Monday after
release; delayed reports absent; `rate_diff = us_3m − jp_3m`; no rate past the last
published month; July 2024 interventions on their day; no intervention amount lost.

> **Lesson 4: merging**
>
> - **Situation:** sources at four frequencies (daily, weekly, monthly, quarterly), each
>   with a publication delay, must sit on one calendar without look-ahead or invented values.
> - **Resolution:** one master calendar; two explicit views of each slow variable
>   (`as of` its period, and `_known` at the time); errors instead of silent drops.
> - **Protocol:** for each column, write in the docstring *what it means on a given day*
>   and *which analysis may use it*. Test the no-look-ahead property directly
>   (value on day *d* = latest release before *d*), on dense samples: sparse samples hid a bug.
> - **Intuition:** `ffill` is the most dangerous line in a pipeline: it is right for
>   "latest known value" and wrong for "value of this period". Ask which one you mean.
> - **Design:** small pure functions (`as_known`, `rates_as_of`, `interventions_daily`)
>   that can be tested on toy data, assembled by one `build_daily`.

## 5. Git: branches, an accident, conflicts and history

### 5.1 The branch map
```text
main ─ d1e5b55 FRED (pushed straight to main, by mistake)
     ─ 139d52e Revert "FRED"            (someone undid it)
     ─ c205c88 Reapply "FRED"            (via PR #24, branch reapply-fred)
     ─ c5cc04a Final data validation      (PR #23, "rebase and merge")
     ─ cbe68ee / 627432f data cleaning     (PR #25, merge commit)

final-validation:  d1e5b55 ─ 8b946c7 validation ─ merge of main
data-cleaning:     built on final-validation (a "stacked" branch)
```

### 5.2 Incident 1: a commit on `main`, reverted
**What happened.** The FRED commit landed directly on `main`. Someone reverted it
(`139d52e`), which removed *all* the FRED work, not just a description. My branch
`final-validation` was built on the FRED commit, so merging `main` into it tried to delete
FRED again: conflicts in `README.md`, `decisions.md`, `config.py`, `fred.py`, and
`deleted by them: tests/test_fred.py`.

**Diagnosis commands.**
```bash
git log --oneline --graph --all -12      # see where branches diverge
git show --stat 139d52e                  # what the revert removed
git diff --stat d1e5b55 fix/fred         # empty = same content in another commit
```

**Fix.** Revert the revert (`git revert 139d52e`): a new commit that puts the FRED work
back. `main` is protected, so a direct push was refused; the fix went through a branch
(`reapply-fred`) and PR #24.

### 5.3 Incident 2: "Which version do I keep?"
During the merge, VS Code showed two versions of `fred.py`. The right one had to be
**proved**, not guessed:
```bash
git diff --cached --stat 8b946c7         # staged result vs my validation commit
git diff --stat d1e5b55 origin/main      # main has the same FRED content as the original
```
Both empty ⇒ the staged result was exactly my branch, and `main` brought nothing new.

**The red Pylance warning (`"pd" is not defined`)** and the "missing pending marks" in
`decisions.md` had the same cause: **unsaved editor tabs** (● on the tab) still holding
an old copy from the conflict. VS Code does not reload a tab that has unsaved changes when
Git rewrites the file. Fix: *File: Revert File*; never save such a tab.

### 5.4 Incident 3: stacked branches and the duplicate commit
`data-cleaning` was created from `final-validation` so cleaning could start while #23
was in review. #23 was merged with **"Rebase and merge"**, which *copies* commits onto
`main` with new ids (`c5cc04a`) and drops merge commits. Git then saw the same rows in
`decisions.md` twice → conflict in #25.

**Fix.** Merge `main` into `data-cleaning`, keep our side of `decisions.md` (it already
contained everything), push. Then merge #25. For a stacked branch whose base was rebased,
prefer "Squash and merge" or a merge commit, never "Rebase and merge" again.

### 5.5 Incident 4: rewriting pushed history
Goal: remove a trailer line from three pushed commit messages (one on `main`).
```bash
git branch backup/<name> <name>                          # safety copy first
git filter-branch --msg-filter "sed '/^Co-Authored-By/d'" -- main final-validation data-cleaning ^139d52e
git push --force-with-lease=main:<old sha> origin main   # refuses if someone pushed meanwhile
```
Steps that failed and why:
- `stale info`: the expected old sha was wrong (the branches had already been pushed once).
- `repository rule violations` on `main`: turning off *Block force pushes* was not enough;
  *Require a pull request* also blocks direct pushes, and the ruleset change had not been **saved**.

Cost: everyone must `git fetch && git reset --hard origin/main`. Only worth it on
unmerged branches or right after the push; an older commit contained in every branch
(`8c53bba`) was left alone.

### 5.6 Branches as a record
Deleting a merged branch removes only its *name*; commits in `main` and pull requests stay.
Because the project is graded on its Git history, branches are **kept** on GitHub.

> **Lesson 5: Git**
>
> - **Situation:** work landed on the wrong branch, was reverted by someone else, and
>   later merges kept conflicting because the same changes existed as different commits.
> - **Resolution:** *diagnose with content comparisons* (`git diff --stat A B`; empty
>   means identical), undo with new commits (`git revert`) rather than deleting history,
>   and route everything to `main` through PRs.
> - **Protocol (daily):**
>   1. `git status` before every `switch`, `pull` or `merge`; commit or `git stash` first.
>   2. Close or revert editor tabs with ● before switching branches.
>   3. Never commit on `main`: `git switch -c <topic>` first.
>   4. Pull `main` into your topic branch often (`git pull origin main`), not only at the end.
> - **Protocol (conflict):** `git status` → for each file ask "which side has *more*
>   of the truth?" → prove it with `git diff --stat` against both parents → resolve →
>   run the tests → commit. Use `git merge --abort` if lost; it is always safe.
> - **Protocol (stacked branches):** base B on A only if B truly depends on A; when A
>   changes, `git merge A` into B; after A is merged, retarget B's PR to `main`.
> - **Intuition:** Git tracks *commits*, not *content*. Two commits with the same content
>   (cherry-pick, rebase-merge, revert-of-revert) look different to Git, and that is
>   where the "impossible" conflicts come from.
> - **Design:** protected `main` + one PR per topic + reviewer who did not write it;
>   `--force-with-lease`, never `--force`; a backup branch before any history rewrite.

## 6. Reusable checklists

### New data source
- [ ] Constants (URL, codes, lag) in `config.py`, with how each was checked
- [ ] Identify by stable keys; check units, encoding, duplicates on download (raise)
- [ ] Index = date the value refers to; `release_date` column with a checked rule
- [ ] Tests on the saved file: shape, ranges, a known historical value, no look-ahead
- [ ] Row(s) in `docs/decisions.md`; README section

### Before trusting a number
- [ ] Is it the right unit and scale?
- [ ] Is there an independent source to compare with (exactly, not on average)?
- [ ] When was it public? Could the model have used it earlier than that?

### Before merging a PR
- [ ] `git pull origin main` into the branch; tests pass (`python -m pytest -q`)
- [ ] Diff shows only this topic's files
- [ ] Merge method: squash or merge commit (not rebase if another branch is stacked on it)

### After merging
- [ ] `git switch main && git pull`; re-fetch sources and rebuild the table if data code changed
- [ ] Tell the team what changed in the data (index, columns, dates)